# 05 - Double DQN Training Loop (Module-First)

Notebook nay goi truc tiep notebook API trong `src.rl.training.notebook_api`.

Ho tro 2 mode:
- pure: train RL tu dau
- warmstart: nap baseline checkpoint + preprocessing artifacts

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to sys.path
cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd] + list(cwd.parents) if (p / 'src').exists()), cwd)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)


from src.ml.artifacts import get_ml_checkpoint_path, get_ml_preprocessing_path
from src.rl.training.notebook_api import run_rl
from src.rl.training.runner import RLTrainingConfig

In [ ]:
ROOT = Path('/workspace/ai-core')
MODE = os.getenv('RL_MODE', 'warmstart').strip().lower()  # pure | warmstart
RUN_ID = os.getenv('RL_RUN_ID', f'notebook_{MODE}_h15')

config = RLTrainingConfig(
    run_id=RUN_ID,
    start_date=os.getenv('RL_START_DATE', '2026-03-25'),
    end_date=os.getenv('RL_END_DATE', '2026-04-16'),
    peak_hours_only=True,
    episodes=int(os.getenv('RL_EPISODES', '24')),
    batch_size=int(os.getenv('RL_BATCH_SIZE', '64')),
    prediction_horizon_minutes=int(os.getenv('RL_PREDICTION_HORIZON_MINUTES', '15')),
    artifacts_path=os.getenv('RL_ARTIFACTS_PATH', str(get_ml_preprocessing_path(run_id='manual_h15'))),
    pretrained_model_path=os.getenv('RL_PRETRAINED_MODEL_PATH', str(get_ml_checkpoint_path(run_id='manual_h15'))),
)

print('MODE  :', MODE)
print('RUN_ID:', RUN_ID)
print('Warmstart artifacts:', config.artifacts_path)
print('Warmstart checkpoint:', config.pretrained_model_path)

In [ ]:
RUN_TRAIN = False

if MODE not in {'pure', 'warmstart'}:
    raise ValueError("MODE must be 'pure' or 'warmstart'")

if RUN_TRAIN:
    result = run_rl(mode=MODE, config=config)
    print('Training done for mode:', result['mode'])
    print('Run ID:', result['run_id'])
    print('Checkpoint:', result['checkpoint_path'])
    print('History   :', result['history_path'])
    print('Metrics   :', result['metrics_path'])
    if result.get('metrics'):
        final_summary = result['metrics'].get('final_summary', {})
        print('Final summary keys:', sorted(final_summary.keys()))
else:
    print('Dry-run: set RUN_TRAIN=True to execute RL training via run_rl.')

In [ ]:
# Optional: inspect RL artifact folders
rl_root = ROOT / 'artifacts' / 'rl'
for subdir in ['checkpoints', 'history', 'metrics']:
    path = rl_root / subdir
    names = [p.name for p in sorted(path.glob('*'))] if path.exists() else []
    print(f'{subdir}:', names[:10])